# What the fixed-core ESP-Gaussian reconstruction measures

Visualise the fitted vorticity, Gaussian weights, and PV-gradient fields using only seabed cells within the `FRAC=1` dynamical core.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
data = ept.load_cache()
gaussian = data[data.method.eq("esp_gaussian_1")].copy()
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
base = gaussian.dropna(subset=["PV_grad_mag"])
ranked = base.assign(rank=base.groupby("Cyc").PV_grad_topo_p90_local_mag.rank(pct=True))
examples = pd.concat([
    part.iloc[(part["rank"]-.9).abs().argsort()[:1]]
    for _, part in ranked.groupby("Cyc")
], ignore_index=True)

In [ ]:
for _, row in examples.iterrows():
    local = ept.local_esp_fields(row, grid, frac=1)
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), constrained_layout=True)
    panels = [("weight","viridis","Gaussian weight"),("zeta","coolwarm","Relative vorticity"),
              ("environment_mag","magma","Environmental log|∇PV|"),("eddy_mag","magma","Internal log|∇PV|")]
    for ax, (column, cmap, title) in zip(axes, panels):
        colour = np.log10(local[column]) if column.endswith("_mag") else local[column]
        artist = ax.scatter(local.x, local.y, c=colour, s=22, cmap=cmap)
        tilt.plot_ellipse(ax, row, grid, frac=1, color="cyan", lw=2, zorder=10)
        ax.set(aspect="equal", title=title, xlabel="x (km)", ylabel="y (km)")
        fig.colorbar(artist, ax=ax, shrink=.75)
    fig.suptitle(f"{row.Cyc}{int(row.Eddy)}, day {int(row.Day)}: only FRAC=1 cells")
    plt.show()

The cyan ellipse is both the radius-of-maximum-velocity boundary and the hard sampling cutoff. The Gaussian remains 0.607 at this boundary; it prioritises the inner core without discarding dynamically relevant outer-core cells.